In [3]:
import nltk
import numpy as np
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.utils import to_categorical

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\kgliv\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [4]:
documents = []
labels = []

for fileid in movie_reviews.fileids():
    documents.append(movie_reviews.raw(fileid))
    labels.append(1 if movie_reviews.categories(fileid)[0] == 'pos' else 0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42)

# Tokenization
vocab_size = 10000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [6]:
# Convert to one-hot (bag-of-words)
X_train_onehot = tokenizer.texts_to_matrix(X_train, mode='binary')
X_test_onehot = tokenizer.texts_to_matrix(X_test, mode='binary')

model_ann_onehot = Sequential([
    Dense(128, activation='relu', input_shape=(vocab_size,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model_ann_onehot.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_ann_onehot.fit(
    X_train_onehot, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

print("ANN One-Hot Accuracy:",
      model_ann_onehot.evaluate(X_test_onehot, y_test)[1])

Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.7578 - loss: 0.4966 - val_accuracy: 0.8375 - val_loss: 0.3571
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9922 - loss: 0.0526 - val_accuracy: 0.8656 - val_loss: 0.3551
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 1.0000 - loss: 0.0066 - val_accuracy: 0.8562 - val_loss: 0.3799
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.8500 - val_loss: 0.3977
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.8500 - val_loss: 0.4135
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8750 - loss: 0.3588
ANN One-Hot Accuracy: 0.875


In [8]:
model_ann_embed = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_ann_embed.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_ann_embed.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

print("ANN Embedding Accuracy:",
      model_ann_embed.evaluate(X_test_pad, y_test)[1])

C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.5414 - loss: 0.6902 - val_accuracy: 0.5406 - val_loss: 0.6797
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9977 - loss: 0.2101 - val_accuracy: 0.6125 - val_loss: 0.6705
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 1.0000 - loss: 0.0058 - val_accuracy: 0.6094 - val_loss: 0.6870
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 0.6187 - val_loss: 0.6972
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.6313 - val_loss: 0.7013
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6250 - loss: 0.7256
ANN Embedding Accuracy: 0.625
